# Proyecto de primera evaluación: Desarrollo y evaluación de modelos de aprendizaje automático para la clasificación de géneros musicales

## 1. Introducción

Para el desarrollo de este proyecto, he utilizado un dataset con cerca de 40.000 registros sobre el análisis de distintas canciones y sus características principales junto con estadísticas sacadas de Spotify.\
La idea es, haciendo uso de ese dataset, realizar una serie de modelos de clasificación y analizar sus predicciones.

[Enlace al origen del dataset](https://www.kaggle.com/datasets/vicsuperman/prediction-of-music-genre)

## 2. Preparación del proyecto

### 2.1. Importación de las dependencias principales a usar
Debido a que tenemos una primera intuición de qué vamos a utilizar para el desarrollo de los modelos, preparamos un primer bloque de código con todas las dependencias principales que vamos a usar:
- La librería **numpy** para realizar operaciones matemáticas y leer y aprovechar sus tipos de datos
- La librería **pandas** para preparar, analizar, limpiar y ordenar el conjunto de datos a leer
- La dependencia **pyplot** de *matplotlib* para hacer representaciones gráficas de los datos analizados
- Distintos datos de la librería **sklearn** para realizar las operaciones con los modelos:
    - De *model_selection*:
        - **train_test_split** para realizar la separación de los datos en conjuntos de entrenamiento y predicción de forma sencilla
        - **cross_val_score** y **cross_validate** para hacer pruebas de validación cruzada con los datos y facilitar las pruebas con los modelos
    - Librerías de procesamiento:
        - **make_pipeline** de la librería *sklearn.pipeline* para habilitar una forma más sencilla de hacer modelos
        - **StandardScaler** de la librería *preprocessing* para escalar los datos correctamente
    - Modelos de aprendizaje:
        - **LogisticRegression** de *sklearn.linear_model* para hacer modelos de regresión logística
        - **KNeighborsClassifier** de *sklearn.neighbors* para hacer modelos K-NN
        - **DecisionTreeClassifier** de *sklearn.tree* para hacer modelos de árboles de decisión
        - **DummyClassifier** de *sklearn.dummy* para hacer un modelo Dummy y probar la fiabilidad de los modelos
    - Métricas de *sklearn.metrics*:
        - **accuracy_score** para sacar la exactitud (accuracy) de las predicciones del modelo
        - **precision_score** para sacar la precisión de las predicciones del modelo
        - **recall_score**
        - **f1_score**
        - **classification_report** para sacar todos los datos previos en formato tabla
        - **confusion_matrix** para sacar la matriz de confusión
        - **ConfusionMatrixDisplay** para hacer una gráfica de la matriz de confusión


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import (train_test_split, cross_val_score, cross_validate)
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.dummy import DummyClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

from sklearn.metrics import (accuracy_score, confusion_matrix, ConfusionMatrixDisplay, precision_score, recall_score, f1_score, classification_report)

# Para hacer desaparecer los warnings de distintas dependencias:
import warnings
warnings.filterwarnings("ignore")

### 2.2. Carga del Dataset

In [ ]:
df_canciones = pd.read_csv("Datasets/music_genre.csv", index_col="instance_id")

,artist_name,track_name,popularity,acousticness,danceability,duration_ms,energy,instrumentalness,key,liveness,loudness,mode,speechiness,tempo,obtained_date,valence,music_genre
instance_id,,,,,,,,,,,,,,,,,
32894.0,Röyksopp,Röyksopp's Night Out,27.0,0.00468,0.652,-1.0,0.941,0.79200,A#,0.115,-5.201,Minor,0.0748,100.889,4-Apr,0.759,Electronic
46652.0,Thievery Corporation,The Shining Path,31.0,0.01270,0.622,218293.0,0.890,0.95000,D,0.124,-7.043,Minor,0.0300,115.00200000000001,4-Apr,0.531,Electronic
30097.0,Dillon Francis,Hurricane,28.0,0.00306,0.620,215613.0,0.755,0.01180,G#,0.534,-4.617,Major,0.0345,127.994,4-Apr,0.333,Electronic
62177.0,Dubloadz,Nitro,34.0,0.02540,0.774,166875.0,0.700,0.00253,C#,0.157,-4.498,Major,0.2390,128.014,4-Apr,0.270,Electronic
24907.0,What So Not,Divide & Conquer,32.0,0.00465,0.638,222369.0,0.587,0.90900,F#,0.157,-6.266,Major,0.0413,145.036,4-Apr,0.323,Electronic
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58878.0,BEXEY,GO GETTA,59.0,0.03340,0.913,-1.0,0.574,0.00000,C#,0.119,-7.022,Major,0.2980,98.02799999999999,4-Apr,0.330,Hip-Hop
43557.0,Roy Woods,Drama (feat. Drake),72.0,0.15700,0.709,251860.0,0.362,0.00000,B,0.109,-9.814,Major,0.0550,122.04299999999999,4-Apr,0.113,Hip-Hop
39767.0,Berner,Lovin' Me (feat. Smiggz),51.0,0.00597,0.693,189483.0,0.763,0.00000,D,0.143,-5.443,Major,0.1460,131.079,4-Apr,0.395,Hip-Hop


## 3. Análisis Explanatorio de Datos

### 3.1. Visión y limpieza general del Dataset

In [4]:
display(df_canciones.info())

<class 'pandas.DataFrame'>
Index: 50005 entries, 32894.0 to 63470.0
Data columns (total 17 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   artist_name       50000 non-null  str    
 1   track_name        50000 non-null  str    
 2   popularity        50000 non-null  float64
 3   acousticness      50000 non-null  float64
 4   danceability      50000 non-null  float64
 5   duration_ms       50000 non-null  float64
 6   energy            50000 non-null  float64
 7   instrumentalness  50000 non-null  float64
 8   key               50000 non-null  str    
 9   liveness          50000 non-null  float64
 10  loudness          50000 non-null  float64
 11  mode              50000 non-null  str    
 12  speechiness       50000 non-null  float64
 13  tempo             50000 non-null  str    
 14  obtained_date     50000 non-null  str    
 15  valence           50000 non-null  float64
 16  music_genre       50000 non-null  str    
dtypes

None

Podemos observar que contamos con 50.005 registros, aunque algunos de ellos cuentan con valores nulos. Vamos a mirar esas columnas para ver qué datos son los restantes

In [13]:
display(df_canciones[df_canciones.isna().any(axis=1)])

,artist_name,track_name,popularity,acousticness,danceability,duration_ms,energy,instrumentalness,key,liveness,loudness,mode,speechiness,tempo,obtained_date,valence,music_genre
instance_id,,,,,,,,,,,,,,,,,
NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Vale, podemos ver que es un caso curioso donde se han generado 5 filas llenas de NaN, por lo que podemos hacer directamente un dropna para quitarlas:

In [14]:
df_canciones = df_canciones.dropna()

display(df_canciones.info())

<class 'pandas.DataFrame'>
Index: 50000 entries, 32894.0 to 63470.0
Data columns (total 17 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   artist_name       50000 non-null  str    
 1   track_name        50000 non-null  str    
 2   popularity        50000 non-null  float64
 3   acousticness      50000 non-null  float64
 4   danceability      50000 non-null  float64
 5   duration_ms       50000 non-null  float64
 6   energy            50000 non-null  float64
 7   instrumentalness  50000 non-null  float64
 8   key               50000 non-null  str    
 9   liveness          50000 non-null  float64
 10  loudness          50000 non-null  float64
 11  mode              50000 non-null  str    
 12  speechiness       50000 non-null  float64
 13  tempo             50000 non-null  str    
 14  obtained_date     50000 non-null  str    
 15  valence           50000 non-null  float64
 16  music_genre       50000 non-null  str    
dtypes

None

Y esto nos deja un dataset con la siguiente estructura:
<table>
<tr><th>Nombre de columna</th><th>Tipo de dato</th><th>Contenido</th><th>Importancia para el análisis</th></tr>
<tr><td>artist_name</td><td>String</td><td>Nombre del artista</td><td>Nula</td></tr>
<tr><td>track_name</td><td>String</td><td>Nombre de la canción</td><td>Nula</td></tr>
<tr><td>popularity</td><td>Float</td><td></td><td></td></tr>
<tr><td>acousticness</td>Float<td></td><td></td><td></td></tr>
<tr><td>danceability</td>Float<td></td><td></td><td></td></tr>
<tr><td>duration_ms</td>Float<td></td><td></td><td></td></tr>
<tr><td>energy</td><td>Float</td><td></td><td></td></tr>
<tr><td>instrumentalness</td><td>Float</td><td></td><td></td></tr>
<tr><td>key</td><td>String (Categórico)</td><td></td><td></td></tr>
<tr><td>liveness</td><td>Float</td><td></td><td></td></tr>
<tr><td>loudness</td><td>Float</td><td></td><td></td></tr>
<tr><td>mode</td><td>String (Categórico)</td><td></td><td></td></tr>
<tr><td>speechiness</td><td>Float</td><td></td><td></td></tr>
<tr><td>tempo</td><td>String (Categórico)</td><td></td><td></td></tr>
<tr><td>obtained_date</td><td>String</td><td></td><td>Nula</td></tr>
<tr><td>valence</td><td></td><td></td><td></td></tr>
<tr><td>music_genre</td><td>String (Categórico)</td><td></td><td><b>Esencial: Valor a predecir</b></td></tr>
</table>

### 3.2. Eliminación de columnas sin importancia

### 3.3. One Hit Encoding: Preparación de los datos

## 4. Preparación y Predicción de modelos

### 4.0. Preparación de conjuntos de entrenamiento y test

### 4.1. Modelo de Regresión Logística

### 4.2. Modelo K-NN

### 4.3. Modelo de Árbol de decisiones

### 4.4. Métricas de evaluación y comparación con Modelo Dummy clasificatorio